In [3]:
import spacy
# Run: python -m spacy download en_core_web_sm first
nlp = spacy.load("en_core_web_sm")

text = "Hello, world! Let's tokenize this sentence."
doc = nlp(text)
tokens = [token.text for token in doc]
# Output: ['Hello', ',', 'world', '!', 'Let', "'s", 'tokenize', 'this', 'sentence', '.']
print(tokens)


['Hello', ',', 'world', '!', 'Let', "'s", 'tokenize', 'this', 'sentence', '.']


In [10]:
import spacy
import re

# pip install spacy transformers tokenizers sentencepiece
# python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")
text = "Hello, world! Let's tokenize this sentence."

print("=" * 55)
print(f"Input: {text}")
print("=" * 55)


# ── 1. WORD-BASED TOKENIZATION ─────────────────────────────
# Splits on whitespace / punctuation. Simple but large vocab.
# Unknown words become <UNK>. Used by: early NLP, spaCy.
print("\n[1] WORD-BASED (spaCy)")
doc = nlp(text)
word_tokens = [token.text for token in doc]
print("Tokens :", word_tokens)
print("Count  :", len(word_tokens))


# ── 2. SUBWORD-BASED TOKENIZATION ─────────────────────────
# Splits rare words into smaller pieces, keeps common words whole.
# Balances vocab size vs. coverage. Used by: BERT, GPT, LLaMA, T5.

# 2a. WordPiece (BERT-style) — prefixes subwords with "##"
print("\n[2a] SUBWORD — WordPiece (BERT)")
from tokenizers import BertWordPieceTokenizer
wp_tokenizer = BertWordPieceTokenizer()
wp_tokenizer.train_from_iterator(
    [text, "tokenizing unseen words like pneumonoultramicroscopicsilicovolcanoconiosis"],
    vocab_size=100, min_frequency=1
)
wp_out = wp_tokenizer.encode(text)
print("Tokens :", wp_out.tokens)
print("Count  :", len(wp_out.tokens))

# 2b. Byte-Pair Encoding / BPE (GPT-style) — merges frequent pairs
print("\n[2b] SUBWORD — BPE (GPT-style)")
from tokenizers import ByteLevelBPETokenizer
bpe_tokenizer = ByteLevelBPETokenizer()
bpe_tokenizer.train_from_iterator(
    [text * 10],   # small corpus just for demo
    vocab_size=100, min_frequency=1, special_tokens=["<unk>"]
)
bpe_out = bpe_tokenizer.encode(text)
print("Tokens :", bpe_out.tokens)
print("Count  :", len(bpe_out.tokens))

# 2c. Using a pretrained BERT tokenizer (real-world example)
print("\n[2c] SUBWORD — Pretrained BERT (transformers)")
from transformers import BertTokenizer
bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
bert_out = bert_tok.tokenize(text)
print("Tokens :", bert_out)
print("Count  :", len(bert_out))

# ✅ 2d. SentencePiece / Unigram — NEW (used by T5, LLaMA, XLNet)
# Language-agnostic, trains directly on raw text with no pre-tokenization.
# Marks the start of each word with a '▁' (underscore) character.
print("\n[2d] SUBWORD — SentencePiece / Unigram (T5, LLaMA)")
import sentencepiece as spm
import tempfile, os

# SentencePiece needs a text file to train from
corpus = "\n".join([text] * 50)        # repeat to meet min token count
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write(corpus)
    tmp_path = f.name

sp_model_prefix = tmp_path.replace(".txt", "")

# Dynamically cap vocab_size to avoid RuntimeError on small corpora
unique_chars = len(set(corpus)) + 4   # +4 for special tokens (pad, unk, bos, eos)
safe_vocab   = min(50, unique_chars)  # never ask for more than the corpus supports

spm.SentencePieceTrainer.train(
    input=tmp_path,
    model_prefix=sp_model_prefix,
    vocab_size=safe_vocab,             # ← dynamic, safe value
    model_type="unigram",              # ← the Unigram algorithm
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)
sp = spm.SentencePieceProcessor(model_file=f"{sp_model_prefix}.model")
sp_tokens = sp.encode(text, out_type=str)
print("Tokens :", sp_tokens)          # words start with ▁
print("Count  :", len(sp_tokens))

# Pretrained example — T5 tokenizer uses SentencePiece internally
print("\n[2e] SUBWORD — Pretrained T5 SentencePiece (transformers)")
from transformers import T5Tokenizer
t5_tok = T5Tokenizer.from_pretrained("t5-small")
t5_out = t5_tok.tokenize(text)
print("Tokens :", t5_out)
print("Count  :", len(t5_out))

# Cleanup temp files
os.remove(tmp_path)
os.remove(f"{sp_model_prefix}.model")
os.remove(f"{sp_model_prefix}.vocab")


# ── 3. CHARACTER-BASED TOKENIZATION ───────────────────────
# Splits into individual characters. Tiny vocab (26+ chars) but
# very long sequences. No unknown words. Used by: CharRNN, some OCR.
print("\n[3] CHARACTER-BASED")
char_tokens = list(text)           # every char including spaces
print("Tokens :", char_tokens)
print("Count  :", len(char_tokens))

# Filter out spaces if needed
char_no_space = [c for c in text if c != " "]
print("(no spaces):", char_no_space)


# ── 4. BYTE-LEVEL TOKENIZATION ────────────────────────────
# Converts text → raw UTF-8 bytes (0–255). Handles ANY language
# or emoji with no <UNK> ever. Used by: GPT-2, ByT5.
print("\n[4] BYTE-LEVEL")
byte_tokens = list(text.encode("utf-8"))   # list of ints 0-255
byte_hex    = [hex(b) for b in byte_tokens]
print("Bytes (int):", byte_tokens)
print("Bytes (hex):", byte_hex)
print("Count      :", len(byte_tokens))


# ── 5. CUSTOM / RULE-BASED TOKENIZATION ───────────────────
# You define the rules via regex or logic. Useful when you need
# domain-specific splitting (code, medical text, hashtags, etc.)
print("\n[5] CUSTOM / RULE-BASED")

# 5a. Split on whitespace only (no punctuation splitting)
simple_tokens = text.split()
print("Simple split :", simple_tokens)

# 5b. Regex: keep words, numbers, punctuation as separate tokens
pattern = r"\w+|[^\w\s]"
regex_tokens = re.findall(pattern, text)
print("Regex tokens :", regex_tokens)

# 5c. Domain example — tokenize a code snippet
code = "for i in range(10): print(i**2)"
code_pattern = r"[a-zA-Z_]\w*|\d+|[^\w\s]|\s+"
code_tokens = [t for t in re.findall(code_pattern, code) if t.strip()]
print("Code tokens  :", code_tokens)


# ── SUMMARY TABLE ─────────────────────────────────────────
print("\n" + "=" * 60)
print(f"{'Type':<22} {'Vocab Size':<14} {'OOV Risk':<12} {'Example Use'}")
print("-" * 60)
rows = [
    ("Word-based",          "Large (50k+)",  "High",   "spaCy, NLTK"),
    ("Subword — WordPiece", "Medium (30k)",  "None",   "BERT, DistilBERT"),
    ("Subword — BPE",       "Medium (50k)",  "None",   "GPT-2, RoBERTa"),
    ("Subword — Unigram",   "Medium (32k)",  "None",   "T5, LLaMA, XLNet"),  # ✅ NEW
    ("Character",           "Tiny (~100)",   "None",   "CharRNN"),
    ("Byte-level",          "Fixed (256)",   "None",   "GPT-2, ByT5"),
    ("Custom/Regex",        "Flexible",      "Varies", "Domain NLP"),
]
for name, vocab, oov, use in rows:
    print(f"{name:<22} {vocab:<14} {oov:<12} {use}")
print("=" * 60)

Input: Hello, world! Let's tokenize this sentence.

[1] WORD-BASED (spaCy)
Tokens : ['Hello', ',', 'world', '!', 'Let', "'s", 'tokenize', 'this', 'sentence', '.']
Count  : 10

[2a] SUBWORD — WordPiece (BERT)
Tokens : ['he', '##llo', ',', 'world', '!', 'le', '##t', "'", 's', 'tokenize', 'th', '##is', 'sen', '##ten', '##ce', '.']
Count  : 16

[2b] SUBWORD — BPE (GPT-style)
Tokens : ['H', 'e', 'l', 'l', 'o', ',', 'Ġ', 'w', 'o', 'r', 'l', 'd', '!', 'Ġ', 'L', 'e', 't', "'", 's', 'Ġ', 't', 'o', 'k', 'e', 'n', 'i', 'z', 'e', 'Ġ', 't', 'h', 'i', 's', 'Ġ', 's', 'e', 'n', 't', 'e', 'n', 'c', 'e', '.']
Count  : 43

[2c] SUBWORD — Pretrained BERT (transformers)
Tokens : ['hello', ',', 'world', '!', 'let', "'", 's', 'token', '##ize', 'this', 'sentence', '.']
Count  : 12

[2d] SUBWORD — SentencePiece / Unigram (T5, LLaMA)
Tokens : ['▁', 'H', 'e', 'l', 'l', 'o', ',', '▁', 'w', 'o', 'r', 'l', 'd', '!', '▁', 'L', 'e', 't', "'", 's', '▁', 't', 'o', 'k', 'en', 'i', 'z', 'e', '▁', 't', 'h', 'i', 's', '▁',